# Agentic AI Notebook

A lightweight agentic application inspired by Claude Code. Given a task, the agent works through three blended phases:

1. **Gather Context** — understand the environment, read files, search for relevant information
2. **Take Action** — execute tools, write/edit files, run commands based on the gathered context
3. **Verify Results** — confirm the actions succeeded and the task is complete

These phases are not rigid stages — they blend together naturally. The agent may gather more context mid-action, or take corrective action during verification.

The system is **expandable** via a plugin architecture for custom **tools** and **skills**.

## 1. Setup & Configuration

In [ ]:
import os
import json
import subprocess
import textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Callable
from enum import Enum

# -- LLM client setup --
# Install: pip install openai
# Works with OpenAI, Azure OpenAI, or any OpenAI-compatible API (e.g. Anthropic via proxy, Ollama)
from openai import OpenAI

# Configure your LLM provider here
LLM_CONFIG = {
    "api_key": os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    "base_url": os.environ.get("OPENAI_BASE_URL", None),  # Set for Azure/Ollama/other providers
    "model": os.environ.get("LLM_MODEL", "gpt-4o"),
}

client = OpenAI(
    api_key=LLM_CONFIG["api_key"],
    base_url=LLM_CONFIG["base_url"],
)

WORKING_DIR = Path.cwd()
print(f"Working directory: {WORKING_DIR}")
print(f"LLM model: {LLM_CONFIG['model']}")

## 2. Core Data Structures

The agent's state is tracked through simple dataclasses. Each tool call, its result, and the current phase are recorded so the agent can reason about what has happened and what to do next.

In [ ]:
class Phase(str, Enum):
    """The three agent phases. These blend together during execution."""
    GATHER = "gather_context"
    ACT = "take_action"
    VERIFY = "verify_results"


@dataclass
class ToolResult:
    """Outcome of a single tool invocation."""
    tool_name: str
    arguments: dict
    output: str
    success: bool
    phase: Phase


@dataclass
class AgentState:
    """Tracks the full lifecycle of an agent run."""
    task: str
    phase: Phase = Phase.GATHER
    history: list[ToolResult] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    done: bool = False
    final_answer: str = ""

    @property
    def summary(self) -> str:
        lines = [f"Task: {self.task}", f"Phase: {self.phase.value}", f"Steps taken: {len(self.history)}"]
        for i, result in enumerate(self.history, 1):
            status = "OK" if result.success else "FAIL"
            lines.append(f"  {i}. [{result.phase.value}] {result.tool_name}({result.arguments}) -> {status}")
        return "\n".join(lines)

## 3. Tool Registry — Extensible Plugin System

Tools are registered with a decorator. Each tool provides:
- A **name** used by the LLM to invoke it
- A **description** the LLM reads to decide when to use it
- A **JSON Schema** for its parameters
- A **Python function** that executes the tool

Adding a new tool is as simple as writing a function and decorating it.

In [ ]:
@dataclass
class ToolDef:
    """Definition of a registered tool."""
    name: str
    description: str
    parameters: dict  # JSON Schema
    func: Callable[..., str]
    phase_hint: Phase | None = None  # Optional: suggest which phase this tool belongs to

    def to_openai_tool(self) -> dict:
        """Convert to the OpenAI function-calling format."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }


class ToolRegistry:
    """Central registry for all available tools."""

    def __init__(self):
        self._tools: dict[str, ToolDef] = {}

    def register(
        self,
        name: str,
        description: str,
        parameters: dict,
        phase_hint: Phase | None = None,
    ):
        """Decorator to register a function as a tool."""
        def decorator(func: Callable[..., str]):
            self._tools[name] = ToolDef(
                name=name,
                description=description,
                parameters=parameters,
                func=func,
                phase_hint=phase_hint,
            )
            return func
        return decorator

    def get(self, name: str) -> ToolDef | None:
        return self._tools.get(name)

    def list_tools(self) -> list[str]:
        return list(self._tools.keys())

    def to_openai_tools(self) -> list[dict]:
        return [t.to_openai_tool() for t in self._tools.values()]

    def execute(self, name: str, arguments: dict) -> tuple[str, bool]:
        """Execute a tool by name. Returns (output, success)."""
        tool = self._tools.get(name)
        if not tool:
            return f"Error: Unknown tool '{name}'", False
        try:
            result = tool.func(**arguments)
            return str(result), True
        except Exception as e:
            return f"Error executing {name}: {e}", False


# Global registry — tools register themselves here
registry = ToolRegistry()
print(f"Tool registry initialized.")

## 4. Built-in Tools

A starter set of tools that cover the basics: reading files, listing directories, searching content, running shell commands, and writing files. These mirror the core capabilities of Claude Code.

In [ ]:
# ---------- Gather-phase tools ----------

@registry.register(
    name="read_file",
    description="Read the contents of a file. Returns the file content as a string.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Path to the file (relative to working dir)"},
        },
        "required": ["path"],
    },
    phase_hint=Phase.GATHER,
)
def read_file(path: str) -> str:
    target = WORKING_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"File not found: {target}")
    content = target.read_text(encoding="utf-8", errors="replace")
    # Truncate very large files to stay within context limits
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + f"\n... [truncated, {len(content)} chars total]"
    return content


@registry.register(
    name="list_directory",
    description="List files and directories at a given path. Returns names with [dir] or [file] markers.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Directory path (relative to working dir). Defaults to '.'"},
        },
        "required": [],
    },
    phase_hint=Phase.GATHER,
)
def list_directory(path: str = ".") -> str:
    target = WORKING_DIR / path
    if not target.is_dir():
        raise NotADirectoryError(f"Not a directory: {target}")
    entries = sorted(target.iterdir())
    lines = []
    for entry in entries:
        if entry.name.startswith("."):
            continue  # skip hidden files
        kind = "[dir] " if entry.is_dir() else "[file]"
        lines.append(f"{kind} {entry.name}")
    return "\n".join(lines) if lines else "(empty directory)"


@registry.register(
    name="search_files",
    description="Search for a text pattern across files. Returns matching lines with file paths and line numbers.",
    parameters={
        "type": "object",
        "properties": {
            "pattern": {"type": "string", "description": "Text or regex pattern to search for"},
            "glob": {"type": "string", "description": "File glob to limit search (e.g. '*.py'). Default: '*'"},
        },
        "required": ["pattern"],
    },
    phase_hint=Phase.GATHER,
)
def search_files(pattern: str, glob: str = "*") -> str:
    import re
    results = []
    for filepath in WORKING_DIR.rglob(glob):
        if not filepath.is_file() or ".git" in filepath.parts:
            continue
        try:
            text = filepath.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        for line_num, line in enumerate(text.splitlines(), 1):
            if re.search(pattern, line):
                rel = filepath.relative_to(WORKING_DIR)
                results.append(f"{rel}:{line_num}: {line.rstrip()}")
        if len(results) >= 50:
            results.append("... (results truncated at 50 matches)")
            break
    return "\n".join(results) if results else f"No matches found for pattern: {pattern}"


# ---------- Action-phase tools ----------

@registry.register(
    name="write_file",
    description="Write content to a file. Creates the file if it doesn't exist, overwrites if it does.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path (relative to working dir)"},
            "content": {"type": "string", "description": "Content to write"},
        },
        "required": ["path", "content"],
    },
    phase_hint=Phase.ACT,
)
def write_file(path: str, content: str) -> str:
    target = WORKING_DIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Wrote {len(content)} chars to {path}"


@registry.register(
    name="edit_file",
    description="Replace a specific string in a file with new content. The old_string must appear exactly once in the file.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path (relative to working dir)"},
            "old_string": {"type": "string", "description": "Exact text to find and replace"},
            "new_string": {"type": "string", "description": "Replacement text"},
        },
        "required": ["path", "old_string", "new_string"],
    },
    phase_hint=Phase.ACT,
)
def edit_file(path: str, old_string: str, new_string: str) -> str:
    target = WORKING_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"File not found: {target}")
    content = target.read_text(encoding="utf-8")
    count = content.count(old_string)
    if count == 0:
        raise ValueError(f"old_string not found in {path}")
    if count > 1:
        raise ValueError(f"old_string appears {count} times in {path} — must be unique")
    new_content = content.replace(old_string, new_string, 1)
    target.write_text(new_content, encoding="utf-8")
    return f"Edited {path}: replaced 1 occurrence"


@registry.register(
    name="run_command",
    description="Run a shell command and return its stdout and stderr. Use for builds, tests, git, etc.",
    parameters={
        "type": "object",
        "properties": {
            "command": {"type": "string", "description": "Shell command to execute"},
        },
        "required": ["command"],
    },
    phase_hint=Phase.ACT,
)
def run_command(command: str) -> str:
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True,
        cwd=str(WORKING_DIR), timeout=60,
    )
    output_parts = []
    if result.stdout.strip():
        output_parts.append(f"STDOUT:\n{result.stdout.strip()}")
    if result.stderr.strip():
        output_parts.append(f"STDERR:\n{result.stderr.strip()}")
    output_parts.append(f"EXIT CODE: {result.returncode}")
    return "\n".join(output_parts)


# ---------- Verify-phase tools ----------

@registry.register(
    name="task_complete",
    description="Signal that the task is finished. Provide a summary of what was done and the outcome.",
    parameters={
        "type": "object",
        "properties": {
            "summary": {"type": "string", "description": "Summary of what was accomplished"},
        },
        "required": ["summary"],
    },
    phase_hint=Phase.VERIFY,
)
def task_complete(summary: str) -> str:
    return f"TASK COMPLETE: {summary}"


print(f"Registered {len(registry.list_tools())} built-in tools: {', '.join(registry.list_tools())}")

## 5. Skills — Higher-Level Composable Actions

Skills are higher-level than tools. A skill is a **multi-step recipe** that composes multiple tool calls to accomplish a common task pattern. Skills are registered separately and the agent can invoke them just like tools.

Think of tools as atoms and skills as molecules.

In [ ]:
@dataclass
class SkillDef:
    """A skill is a multi-step recipe that composes tools."""
    name: str
    description: str
    parameters: dict
    func: Callable[..., str]  # Receives (registry, **kwargs) so it can call tools


class SkillRegistry:
    """Registry for skills. Skills are exposed as tools to the LLM."""

    def __init__(self, tool_registry: ToolRegistry):
        self._skills: dict[str, SkillDef] = {}
        self._tool_registry = tool_registry

    def register(self, name: str, description: str, parameters: dict):
        """Decorator to register a skill."""
        def decorator(func: Callable):
            self._skills[name] = SkillDef(
                name=name,
                description=description,
                parameters=parameters,
                func=func,
            )
            # Also register as a tool so the LLM can call it
            self._tool_registry._tools[name] = ToolDef(
                name=name,
                description=f"[SKILL] {description}",
                parameters=parameters,
                func=lambda **kwargs: func(self._tool_registry, **kwargs),
            )
            return func
        return decorator

    def list_skills(self) -> list[str]:
        return list(self._skills.keys())


skills = SkillRegistry(registry)


# ---------- Built-in Skills ----------

@skills.register(
    name="summarize_project",
    description="Analyze the project structure and produce a summary: directory tree, key files, languages used, and purpose.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Root path to analyze. Defaults to '.'"},
        },
        "required": [],
    },
)
def summarize_project(tool_reg: ToolRegistry, path: str = ".") -> str:
    # Step 1: List the directory
    listing, _ = tool_reg.execute("list_directory", {"path": path})
    # Step 2: Look for common project files
    key_files = []
    for name in ["README.md", "pyproject.toml", "setup.py", "package.json", "Cargo.toml", "go.mod", "Makefile"]:
        full = Path(WORKING_DIR) / path / name
        if full.is_file():
            key_files.append(name)
    # Step 3: Count files by extension
    ext_counts: dict[str, int] = {}
    for fp in (WORKING_DIR / path).rglob("*"):
        if fp.is_file() and ".git" not in fp.parts:
            ext = fp.suffix or "(no extension)"
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
    ext_summary = ", ".join(f"{ext}: {c}" for ext, c in sorted(ext_counts.items(), key=lambda x: -x[1])[:10])
    return (
        f"Directory listing:\n{listing}\n\n"
        f"Key project files found: {', '.join(key_files) if key_files else 'none'}\n"
        f"File types: {ext_summary}\n"
        f"Total files: {sum(ext_counts.values())}"
    )


@skills.register(
    name="find_and_replace",
    description="Search for a pattern across the project and replace all occurrences. Returns a report of changes made.",
    parameters={
        "type": "object",
        "properties": {
            "search_pattern": {"type": "string", "description": "Text to search for"},
            "replacement": {"type": "string", "description": "Text to replace with"},
            "glob": {"type": "string", "description": "File glob to limit scope (e.g. '*.py')"},
        },
        "required": ["search_pattern", "replacement"],
    },
)
def find_and_replace(tool_reg: ToolRegistry, search_pattern: str, replacement: str, glob: str = "*") -> str:
    changes = []
    for filepath in WORKING_DIR.rglob(glob):
        if not filepath.is_file() or ".git" in filepath.parts:
            continue
        try:
            content = filepath.read_text(encoding="utf-8")
        except Exception:
            continue
        if search_pattern in content:
            count = content.count(search_pattern)
            new_content = content.replace(search_pattern, replacement)
            filepath.write_text(new_content, encoding="utf-8")
            rel = filepath.relative_to(WORKING_DIR)
            changes.append(f"{rel}: {count} replacement(s)")
    if not changes:
        return f"No occurrences of '{search_pattern}' found."
    return f"Replaced '{search_pattern}' with '{replacement}' in {len(changes)} file(s):\n" + "\n".join(changes)


print(f"Registered {len(skills.list_skills())} built-in skills: {', '.join(skills.list_skills())}")
print(f"Total available tools (tools + skills): {len(registry.list_tools())}")

## 6. The Agent Loop

The agent loop sends the task to the LLM, processes tool calls, feeds results back, and repeats until the LLM signals completion. The three phases blend naturally — the system prompt guides the LLM through them, but it can move freely between phases as needed.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are an autonomous agent that completes software engineering tasks.
    You work through three blended phases:

    1. GATHER CONTEXT — Read files, list directories, search code to understand what you're working with.
    2. TAKE ACTION — Write files, edit code, run commands to make the required changes.
    3. VERIFY RESULTS — Re-read files, run tests, or check output to confirm your work is correct.

    These phases are not rigid. You may gather more context while acting, or take corrective
    action during verification. Move fluidly between phases as the task requires.

    Guidelines:
    - Always read a file before editing it.
    - After making changes, verify them (re-read the file, run tests, etc.).
    - Use the simplest approach that works.
    - When done, call 'task_complete' with a clear summary.
    - If you get stuck, explain what you tried and what blocked you.
""")

MAX_TURNS = 20  # Safety limit to prevent infinite loops


def infer_phase(tool_name: str) -> Phase:
    """Heuristically determine which phase a tool call belongs to."""
    tool_def = registry.get(tool_name)
    if tool_def and tool_def.phase_hint:
        return tool_def.phase_hint
    return Phase.ACT  # Default


def run_agent(task: str, verbose: bool = True) -> AgentState:
    """Run the agent loop to completion."""
    state = AgentState(task=task)
    state.messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    tools_spec = registry.to_openai_tools()

    for turn in range(MAX_TURNS):
        if verbose:
            print(f"\n{'='*60}")
            print(f"Turn {turn + 1} | Phase: {state.phase.value}")
            print(f"{'='*60}")

        # Call the LLM
        response = client.chat.completions.create(
            model=LLM_CONFIG["model"],
            messages=state.messages,
            tools=tools_spec,
            tool_choice="auto",
        )

        message = response.choices[0].message
        state.messages.append(message.model_dump())

        # If the LLM produced text output, display it
        if message.content:
            if verbose:
                print(f"\nAgent: {message.content}")

        # If no tool calls, the agent is done talking
        if not message.tool_calls:
            state.done = True
            state.final_answer = message.content or ""
            break

        # Process each tool call
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            # Track phase transitions
            new_phase = infer_phase(fn_name)
            if new_phase != state.phase:
                state.phase = new_phase
                if verbose:
                    print(f"\n>> Phase transition -> {state.phase.value}")

            if verbose:
                print(f"\n  Tool: {fn_name}")
                print(f"  Args: {json.dumps(fn_args, indent=2)[:200]}")

            # Execute
            output, success = registry.execute(fn_name, fn_args)

            if verbose:
                display = output[:300] + ("..." if len(output) > 300 else "")
                print(f"  Result ({'OK' if success else 'FAIL'}): {display}")

            # Record
            state.history.append(ToolResult(
                tool_name=fn_name,
                arguments=fn_args,
                output=output,
                success=success,
                phase=state.phase,
            ))

            # Check for task_complete
            if fn_name == "task_complete":
                state.done = True
                state.final_answer = fn_args.get("summary", output)

            # Feed result back to LLM
            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

        if state.done:
            break

    if verbose:
        print(f"\n{'='*60}")
        print(f"Agent finished in {len(state.history)} steps.")
        if state.final_answer:
            print(f"\nFinal answer: {state.final_answer}")
        print(f"{'='*60}")

    return state


print("Agent loop ready.")

## 7. Extending the Agent — Custom Tools & Skills

Adding your own tools and skills is straightforward. Use the `@registry.register` decorator for tools and `@skills.register` for skills.

Below are examples you can use as templates.

In [ ]:
# ===== EXAMPLE: Custom Tool — Count Lines =====

@registry.register(
    name="count_lines",
    description="Count the number of lines in a file.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path"},
        },
        "required": ["path"],
    },
    phase_hint=Phase.GATHER,
)
def count_lines(path: str) -> str:
    target = WORKING_DIR / path
    lines = target.read_text(encoding="utf-8", errors="replace").splitlines()
    return f"{path}: {len(lines)} lines"


# ===== EXAMPLE: Custom Skill — Run Tests and Report =====

@skills.register(
    name="run_tests",
    description="Detect the test framework in use, run the test suite, and return a pass/fail summary.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
    },
)
def run_tests(tool_reg: ToolRegistry) -> str:
    # Detect test runner
    runners = [
        ("pyproject.toml", "pytest"),
        ("package.json", "npm test"),
        ("Makefile", "make test"),
        ("Cargo.toml", "cargo test"),
    ]
    for config_file, cmd in runners:
        if (WORKING_DIR / config_file).exists():
            output, success = tool_reg.execute("run_command", {"command": cmd})
            return f"Ran '{cmd}':\n{output}"
    return "No test runner detected. Checked for: pyproject.toml, package.json, Makefile, Cargo.toml"


print(f"Custom extensions loaded. Total tools: {len(registry.list_tools())}")
print(f"All available: {', '.join(registry.list_tools())}")

## 8. Run the Agent

Give the agent a task and watch it work through the three phases. Try different tasks to see how it blends context gathering, action, and verification.

In [ ]:
# -- Try it out! Change the task to whatever you'd like. --

task = """
Explore this project directory. List the files, read any interesting ones,
and give me a summary of what this project is about.
"""

state = run_agent(task.strip())

In [ ]:
# -- Review the agent's execution trace --

print(state.summary)

## 9. Adding Your Own Tools — Template

Copy and customize the template below to add your own tools. Re-run this cell and the agent will pick up the new tool on its next run.

In [ ]:
# ===== YOUR CUSTOM TOOL TEMPLATE =====
#
# @registry.register(
#     name="my_tool_name",
#     description="What this tool does — the LLM reads this to decide when to use it.",
#     parameters={
#         "type": "object",
#         "properties": {
#             "param1": {"type": "string", "description": "Description of param1"},
#             "param2": {"type": "integer", "description": "Description of param2"},
#         },
#         "required": ["param1"],
#     },
#     phase_hint=Phase.ACT,  # or Phase.GATHER, Phase.VERIFY
# )
# def my_tool_name(param1: str, param2: int = 10) -> str:
#     # Your tool logic here
#     return f"Result: {param1}, {param2}"


# ===== YOUR CUSTOM SKILL TEMPLATE =====
#
# @skills.register(
#     name="my_skill_name",
#     description="A multi-step operation that composes existing tools.",
#     parameters={
#         "type": "object",
#         "properties": {
#             "target": {"type": "string", "description": "What to operate on"},
#         },
#         "required": ["target"],
#     },
# )
# def my_skill_name(tool_reg: ToolRegistry, target: str) -> str:
#     # Compose multiple tool calls
#     step1, _ = tool_reg.execute("list_directory", {"path": target})
#     step2, _ = tool_reg.execute("search_files", {"pattern": "TODO", "glob": "*.py"})
#     return f"Directory:\n{step1}\n\nTODOs found:\n{step2}"

print("Templates ready. Uncomment and customize to add your own tools and skills.")